In [ ]:
import pandas as pd 
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np 
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

In [ ]:
df = pd.read_excel("data/donnees_immobilieres.xlsx")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.skew(numeric_only=True)


In [ ]:
df.kurtosis(numeric_only=True)

**VISUALISATION**

In [ ]:
sns.set_style("ticks")

In [ ]:
cols_to_plot = [
    "Surface_m2",
    "Chambres",
    "Distance_centre_km",
    "Etage",
    "Qualite_ecole",
    "Revenu_median_quartier",
    "Distance_universite",
    "Prix_milliers_euros"
]

n_cols = 3
n_rows = (len(cols_to_plot) + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    nrows=n_rows,
    ncols=n_cols,
    figsize=(14, 4 * n_rows)
)

axes = axes.flatten()

for ax, col in zip(axes, cols_to_plot):
    sns.boxplot(data=df, y=col, ax=ax)
    ax.set_title(col, fontsize=11, weight="bold")
    ax.set_ylabel("")

# Masquer les axes inutilisés
for ax in axes[len(cols_to_plot):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig("boxplots.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(
    nrows=n_rows,
    ncols=n_cols,
    figsize=(14, 4 * n_rows)
)

axes = axes.flatten()

for ax, col in zip(axes, cols_to_plot):
    data = df[col].dropna()

    # Histogramme + KDE
    sns.histplot(data, bins=30, kde=True, ax=ax)

    # Calcul des outliers (IQR)
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Lignes des seuils
    ax.axvline(lower, linestyle="--", linewidth=2, label="Outlier seuil")
    ax.axvline(upper, linestyle="--", linewidth=2)

    ax.set_title(col, fontsize=11, weight="bold")
    ax.set_ylabel("")

# Masquer les axes inutilisés
for ax in axes[len(cols_to_plot):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig("hist.png", dpi=300, bbox_inches="tight")
plt.show()


**ANALYSE DE CORRELATION**

In [ ]:
continuous_cols = [
    "Surface_m2",
    "Distance_centre_km",
    "Revenu_median_quartier",
    "Distance_universite",
    "Prix_milliers_euros",
    "Annee_vente",
    "Annee_construction"
]

In [ ]:
corr = df[continuous_cols].corr(numeric_only=True)  # par défaut: Pearson

In [ ]:
corr

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,        # affiche les valeurs
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5
)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

**MODELE SIMPLE**

In [ ]:
X = df["Surface_m2"]
X= sm.add_constant(X)
y = df["Prix_milliers_euros"]

In [ ]:
model = sm.OLS(y, X)
results = model.fit()

In [ ]:
print(results.summary())

**MODELE MULTIPLE**

In [ ]:
X = df[[
    "Surface_m2",
    "Chambres",
    "Annee_construction",
    "Distance_centre_km",
    "Etage",
    "Ascenseur"
]]



In [ ]:
X = sm.add_constant(X)
y = df["Prix_milliers_euros"]

In [ ]:
model = sm.OLS(y, X)
results = model.fit()

In [ ]:
print(results.summary())

**MODELE SEMI-LOG (LOG-LIN)**

In [ ]:
y_log = np.log(df["Prix_milliers_euros"])

model_semi_log = sm.OLS(y_log, X)
results_semi_log = model_semi_log.fit()

print(results_semi_log.summary())


**MODELE LOG-LOG**

In [ ]:
X = df[
    ["Surface_m2", "Chambres", "Annee_construction",
     "Distance_centre_km", "Etage", "Ascenseur"]
].copy()

X["log_Surface_m2"] = np.log(X["Surface_m2"])
X["log_Distance_centre_km"] = np.log(X["Distance_centre_km"])

X = X.drop(columns=["Surface_m2", "Distance_centre_km"])

X = sm.add_constant(X)

y = np.log(df["Prix_milliers_euros"])

model_log_log = sm.OLS(y, X)
results_log_log = model_log_log.fit()

print(results_log_log.summary())

In [ ]:
X = df[['Surface_m2',
        'Chambres',
        'Annee_construction',
        'Distance_centre_km',
        'Etage',
        'Ascenseur']]
# ajouter la constante
X = sm.add_constant(X)

In [ ]:
vif = pd.DataFrame()
vif["Variable"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif)

In [ ]:
t_stat = results_semi_log.tvalues['Distance_centre_km']
p_two_sided = results_semi_log.pvalues['Distance_centre_km']

t_stat, p_two_sided


In [ ]:
p_one_sided = p_two_sided / 2
p_one_sided

In [ ]:
X_full = df[[
    "Surface_m2",
    "Chambres",
    "Annee_construction",
    "Distance_centre_km",
    "Etage",
    "Ascenseur",
    "Qualite_ecole",
    "Revenu_median_quartier"
]]

X_full = sm.add_constant(X_full)

results_semi_log_full = sm.OLS(y_log, X_full).fit()
print(results_semi_log_full.summary())


In [ ]:
print(results_semi_log_full.compare_f_test(results_semi_log))

**STABILITE STRUCTURELLE**

In [ ]:
df["COVID"] = (df["Annee_vente"] >= 2020).astype(int)

In [ ]:
df.head()

In [ ]:
X_covid = df[[
    "Surface_m2",
    "Chambres",
    "Annee_construction",
    "Distance_centre_km",
    "Etage",
    "Ascenseur",
    "COVID"
]]

X_covid = sm.add_constant(X_covid)

results_covid = sm.OLS(y_log, X_covid).fit()
print(results_covid.summary())


In [ ]:
df["COVID_Surface"] = df["COVID"] * df["Surface_m2"]
df["COVID_Distance"] = df["COVID"] * df["Distance_centre_km"]

df.head()


In [ ]:
X_interact = df[[
    "Surface_m2",
    "Chambres",
    "Annee_construction",
    "Distance_centre_km",
    "Etage",
    "Ascenseur",
    "COVID",
    "COVID_Surface",
    "COVID_Distance"
]]

X_interact = sm.add_constant(X_interact)

results_interact = sm.OLS(y_log, X_interact).fit()
print(results_interact.summary())

In [ ]:
residuals = results_semi_log.resid
fitted = results_semi_log.fittedvalues


In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(
    x=fitted,
    y=residuals,
    lowess=True,
    scatter_kws={"alpha": 0.6},
    line_kws={"color": "red"}
)

plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel("Valeurs ajustées (log(prix))", fontsize=12)
plt.ylabel("Résidus", fontsize=12)
plt.title("Résidus vs valeurs ajustées avec lissage LOWESS", fontsize=14)
plt.grid(True, alpha=0.3)
plt.savefig("residuals_vs_fitted.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
residuals = results_semi_log.resid
exog = results_semi_log.model.exog

bp_test = het_breuschpagan(residuals, exog)

bp_labels = ['LM statistic', 'LM p-value', 'F statistic', 'F p-value']
dict(zip(bp_labels, bp_test))